# Treino - Densenet - Modelo de classificação de Coral-Sol

Autor:  Viviane da Rosa Sommer

Atualizado: 16/12/2024

## Objetivo:
Notebook para fazer o treino do primeiro modelo de classificação de coral, com patches das imagens usados no modelo de segmentação (Lotes 1, 2 e 3 - Limpos).

## Importações Necessárias

In [ ]:
!pip install opencv-python-headless
!pip install --upgrade tensorflow
!pip install --upgrade keras
import tensorflow as tf
import keras
from keras import layers, models
from keras.applications import DenseNet121
from keras.preprocessing import image_dataset_from_directory
from keras.callbacks import EarlyStopping, ModelCheckpoint

## Declaração de Constantes

In [ ]:
train_dir = "Datasets/Dataset-Classificacao-Patchs-V1/train"
val_dir = "Datasets/Dataset-Classificacao-Patchs-V1/val"
test_dir = "Datasets/Dataset-Classificacao-Patchs-V1/test"
weights_path = 'densenet121_weights_tf_dim_ordering_tf_kernels_notop.h5'

batch_size = 32
image_size = (128, 128)  

## Carregar Dataset

In [ ]:
train_dataset = image_dataset_from_directory(
    train_dir,
    image_size=image_size,
    batch_size=batch_size,
    label_mode="binary" 
)

val_dataset = image_dataset_from_directory(
    val_dir,
    image_size=image_size,
    batch_size=batch_size,
    label_mode="binary"
)

test_dataset = image_dataset_from_directory(
    test_dir,
    image_size=image_size,
    batch_size=batch_size,
    label_mode="binary"
)

In [ ]:
preprocess_input = keras.applications.densenet.preprocess_input

train_dataset = train_dataset.map(lambda x, y: (preprocess_input(x), y))
val_dataset = val_dataset.map(lambda x, y: (preprocess_input(x), y))
test_dataset = test_dataset.map(lambda x, y: (preprocess_input(x), y))


## Construção do Modelo

In [ ]:
weights_path = 'densenet121_weights_tf_dim_ordering_tf_kernels_notop.h5'

base_model = DenseNet121(
    weights=weights_path, 
    include_top=False, 
    input_shape=(128, 128, 3) 
)
base_model.trainable = False

In [ ]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid") 
])

In [ ]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy', keras.metrics.AUC()])

## Treino do Modelo

In [ ]:
epochs = 100
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint('best_weights.keras', monitor='val_loss', save_best_only=True, mode='min')

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=epochs,
    callbacks=[early_stopping, checkpoint]
)

## Métricas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc


test_loss, test_acc, test_auc = model.evaluate(test_dataset)
print(f"Test Loss: {test_loss:.2f}")
print(f"Test Accuracy: {test_acc:.2f}")
print(f"Test AUC: {test_auc:.2f}")

y_true = []
y_pred = []

for images, labels in test_dataset:
    y_true.extend(labels.numpy())  
    y_pred.extend(model.predict(images)) 
y_true = np.array(y_true)
y_pred = np.array(y_pred)

y_pred_bin = (y_pred > 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_true, y_pred_bin))

cm = confusion_matrix(y_true, y_pred_bin)
print("\nConfusion Matrix:")
plt.figure(figsize=(6, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.xticks([0, 1], ['Negative', 'Positive'])
plt.yticks([0, 1], ['Negative', 'Positive'])
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", va="center", color="white" if cm[i, j] > thresh else "black")

plt.tight_layout()
plt.show()


fpr, tpr, _ = roc_curve(y_true, y_pred)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc='lower right')
plt.show()